# 3. Desempenho e Complexidade de Algoritmos
Neste notebook, implementamos algoritmos clássicos de grafos e confrontamos a complexidade teórica com o tempo de execução empírico na rede real do *Memetracker*.

In [ ]:
import networkx as nx
import pandas as pd
import time
import numpy as np
import scipy.stats as st
import random

# Carregando os dados e preparando a Maior Componente Conexa (LCC)
print("Carregando arestas...")
df = pd.read_csv('../data/edges_2days.csv', on_bad_lines='skip')
G_dir = nx.from_pandas_edgelist(df, source='source', target='target', create_using=nx.DiGraph())

G_undir = G_dir.to_undirected()
G_undir.remove_edges_from(nx.selfloop_edges(G_undir))

connected_components = list(nx.connected_components(G_undir))
lcc_nodes = max(connected_components, key=len)
LCC = G_undir.subgraph(lcc_nodes).copy()

print(f"Grafo Direcionado Original: {G_dir.number_of_nodes()} V, {G_dir.number_of_edges()} E")
print(f"Maior Componente Conexa (LCC): {LCC.number_of_nodes()} V, {LCC.number_of_edges()} E")

### Ferramentas Estatísticas
Funções para realizar as rodadas experimentais e calcular o tempo médio, desvio padrão e Intervalos de Confiança (Normal ou t-Student).

In [ ]:
def calc_stats(times):
    n = len(times)
    mean = np.mean(times)
    std = np.std(times, ddof=1) if n > 1 else 0.0
    
    alpha = 0.05
    if n >= 30:
        z = st.norm.ppf(1 - alpha/2)
        margin = z * (std / np.sqrt(n)) if n > 0 else 0
        dist_name = "Normal (Z)"
    else:
        t = st.t.ppf(1 - alpha/2, n-1) if n > 1 else 0
        margin = t * (std / np.sqrt(n)) if n > 0 else 0
        dist_name = "t-Student"
        
    return mean, std, margin, dist_name, n

def run_experiment(name, func, n_runs):
    print(f"Iniciando: {name} ({n_runs} rodadas)")
    times = []
    for i in range(n_runs):
        start = time.time()
        func(i)
        times.append(time.time() - start)
        
    mean, std, margin, dist_name, n = calc_stats(times)
    
    print(f"[Concluído] {name}")
    print(f"   Distribuição: {dist_name} (n={n})")
    print(f"   Média de Tempo: {mean:.5f}s")
    print(f"   Desvio Padrão:  {std:.5f}s")
    print(f"   IC (95%):       {mean:.5f}s +- {margin:.5f}s  [{mean-margin:.5f}s, {mean+margin:.5f}s]")
    print("-" * 60)

### Buscas em Grafos
Comparamos a Busca em Largura (BFS) e Busca em Profundidade (DFS) na Maior Componente Conexa.

In [ ]:
nodes_list = list(LCC.nodes())
random.seed(42)
sources_30 = random.sample(nodes_list, 30)

run_experiment("Busca em Largura (BFS)", lambda i: list(nx.bfs_edges(LCC, sources_30[i])), 30)
run_experiment("Busca em Profundidade (DFS)", lambda i: list(nx.dfs_edges(LCC, sources_30[i])), 30)
run_experiment("Verificação de Eulerianidade", lambda i: nx.is_eulerian(G_undir), 30)

### Árvores Geradoras Mínimas (AGM) e Componentes Fortemente Conexas
Avaliamos o desempenho de Prim e Kruskal. Também rodamos o Algoritmo de Tarjan no grafo *direcionado* original.

In [ ]:
run_experiment("AGM - Algoritmo de Prim", lambda i: nx.minimum_spanning_tree(LCC, algorithm='prim'), 30)
run_experiment("AGM - Algoritmo de Kruskal", lambda i: nx.minimum_spanning_tree(LCC, algorithm='kruskal'), 30)
run_experiment("Algoritmo de Tarjan (SCC)", lambda i: list(nx.strongly_connected_components(G_dir)), 30)

### Caminhos Mínimos em Subgrafos
Devido à complexidade cúbica do Floyd-Warshall e aos gargalos do Dijkstra, extraímos subgrafos localizados para poder comparar o tempo de execução de Dijkstra e Bellman-Ford, e uma micro-rede para o Floyd-Warshall.

In [ ]:
sub_5k_nodes = list(nx.bfs_tree(LCC, sources_30[0], depth_limit=3).nodes())[:5000]
sub_5k = LCC.subgraph(sub_5k_nodes).copy()
print(f"Subgrafo 5K: {sub_5k.number_of_nodes()} V, {sub_5k.number_of_edges()} E")
sources_sub_30 = random.sample(list(sub_5k.nodes()), 30)

sub_200_nodes = list(nx.bfs_tree(LCC, sources_30[0], depth_limit=2).nodes())[:200]
sub_200 = LCC.subgraph(sub_200_nodes).copy()
print(f"Subgrafo 200: {sub_200.number_of_nodes()} V, {sub_200.number_of_edges()} E\n")

run_experiment("Caminhos Mínimos: BFS (Baseline 5K)", lambda i: list(nx.bfs_edges(sub_5k, sources_sub_30[i])), 30)
run_experiment("Caminhos Mínimos: Dijkstra (5K)", lambda i: nx.single_source_dijkstra_path_length(sub_5k, sources_sub_30[i], weight=None), 30)
run_experiment("Caminhos Mínimos: Bellman-Ford (5K)", lambda i: nx.single_source_bellman_ford_path_length(sub_5k, sources_sub_30[i], weight=None), 30)

run_experiment("Floyd-Warshall (APSP 200)", lambda i: nx.floyd_warshall(sub_200, weight=None), 10)